# TP3 - Exploración para detección del logo Coca-Cola

Este notebook lo uso como cuaderno de trabajo para probar las herramientas vistas en clase antes de dejar una versión final.

La idea es revisar:

- Template Matching con distintos métodos.
- Diferencia entre trabajar en gris y sobre bordes.
- Cambios de escala con `cv.resize`.
- Detección múltiple con umbral.
- Eliminación de cajas repetidas mediante IoU / supresión de no máximos.

No busco que este notebook sea la entrega final, sino justificar las decisiones que después quedan fijas en el notebook de entrega.


In [ ]:
import os
import glob
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt


## 1. Rutas

Si el notebook se ejecuta en la misma carpeta que `images/` y `template/`, toma esas rutas.

También dejo una alternativa para Colab o el entorno actual.


In [ ]:
base_dir = '.'

if not os.path.exists(os.path.join(base_dir, 'images')):
    base_dir = '/mnt/data/tp3data'

images_dir = os.path.join(base_dir, 'images')
template_path = os.path.join(base_dir, 'template', 'pattern.png')

image_paths = sorted(glob.glob(os.path.join(images_dir, '*')))

print('Template:', template_path)
print('Cantidad de imágenes:', len(image_paths))
for p in image_paths:
    print('-', os.path.basename(p))


## 2. Visualización del template y de las imágenes


In [ ]:
template_color = cv.imread(template_path)
template_gray = cv.cvtColor(template_color, cv.COLOR_BGR2GRAY)

plt.figure(figsize=(7, 3))
plt.imshow(cv.cvtColor(template_color, cv.COLOR_BGR2RGB))
plt.title('Template')
plt.axis('off')
plt.show()


In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 11))
axes = axes.ravel()

for i, path in enumerate(image_paths):
    img = cv.imread(path)
    axes[i].imshow(cv.cvtColor(img, cv.COLOR_BGR2RGB))
    axes[i].set_title(os.path.basename(path))
    axes[i].axis('off')

for j in range(len(image_paths), len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()


## 3. Primera prueba: métodos de Template Matching

En clase se probaron varios métodos de `cv.matchTemplate`.

Para esta exploración comparo tres métodos normalizados:

- `TM_CCOEFF_NORMED`
- `TM_CCORR_NORMED`
- `TM_SQDIFF_NORMED`

Para los dos primeros interesa el máximo.  
Para `TM_SQDIFF_NORMED` interesa el mínimo.


In [ ]:
imagen_prueba = cv.imread(image_paths[0])
imagen_prueba_gray = cv.cvtColor(imagen_prueba, cv.COLOR_BGR2GRAY)

methods = {
    'TM_CCOEFF_NORMED': cv.TM_CCOEFF_NORMED,
    'TM_CCORR_NORMED': cv.TM_CCORR_NORMED,
    'TM_SQDIFF_NORMED': cv.TM_SQDIFF_NORMED
}

for nombre, method in methods.items():
    if (template_gray.shape[0] >= imagen_prueba_gray.shape[0] or
        template_gray.shape[1] >= imagen_prueba_gray.shape[1]):
        continue

    res = cv.matchTemplate(imagen_prueba_gray, template_gray, method)
    min_val, max_val, min_loc, max_loc = cv.minMaxLoc(res)

    if method == cv.TM_SQDIFF_NORMED:
        score = 1 - min_val
        loc = min_loc
    else:
        score = max_val
        loc = max_loc

    print(nombre, '- score:', round(float(score), 4), '- ubicación:', loc)


## 4. Preprocesamiento con Canny

En clase se mostró que una forma de reducir falsos positivos es comparar contornos en lugar de trabajar directamente con intensidades.

Para no ajustar los umbrales de Canny imagen por imagen, los calculo a partir de la mediana de intensidad.


In [ ]:
def canny_simple(gray):
    mediana = np.median(gray)

    limite_inferior = int(max(0, 0.67 * mediana))
    limite_superior = int(min(255, 1.33 * mediana))

    return cv.Canny(gray, limite_inferior, limite_superior)


In [ ]:
template_edges = canny_simple(template_gray)

img = cv.imread(image_paths[0])
gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
edges = canny_simple(gray)

plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.imshow(gray, cmap='gray')
plt.title('Imagen en gris')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(edges, cmap='gray')
plt.title('Bordes imagen')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(template_edges, cmap='gray')
plt.title('Bordes template')
plt.axis('off')

plt.tight_layout()
plt.show()


## 5. Búsqueda en distintas escalas

Template Matching solo contempla desplazamientos.  
Como en las imágenes el logo aparece con distintos tamaños, genero varias escalas del template usando `cv.resize`.

El rango se define una sola vez y se usa igual para todas las imágenes.


In [ ]:
scales = np.linspace(0.10, 1.60, 40)

print('Cantidad de escalas:', len(scales))
print('Escala mínima:', scales[0])
print('Escala máxima:', scales[-1])


In [ ]:
def mejor_coincidencia_multiescala(gray, template_gray, usar_bordes=True):
    if usar_bordes:
        imagen_busqueda = canny_simple(gray)
        template_base = canny_simple(template_gray)
    else:
        imagen_busqueda = gray
        template_base = template_gray

    mejor = None

    for scale in scales:
        template_resized = cv.resize(
            template_base,
            None,
            fx=scale,
            fy=scale,
            interpolation=cv.INTER_AREA
        )

        h, w = template_resized.shape

        if h < 8 or w < 8:
            continue

        if h >= imagen_busqueda.shape[0] or w >= imagen_busqueda.shape[1]:
            continue

        res = cv.matchTemplate(
            imagen_busqueda,
            template_resized,
            cv.TM_CCOEFF_NORMED
        )

        _, max_val, _, max_loc = cv.minMaxLoc(res)

        actual = {
            'score': float(max_val),
            'scale': float(scale),
            'loc': max_loc,
            'w': w,
            'h': h
        }

        if mejor is None or actual['score'] > mejor['score']:
            mejor = actual

    return mejor


In [ ]:
print('Mejor coincidencia por imagen')
print('--------------------------------')

for path in image_paths:
    img = cv.imread(path)
    gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)

    mejor = mejor_coincidencia_multiescala(
        gray,
        template_gray,
        usar_bordes=True
    )

    print(
        os.path.basename(path),
        '- score:', round(mejor['score'], 3),
        '- escala:', round(mejor['scale'], 3)
    )


## 6. Detecciones múltiples

Para `coca_multi.png` no alcanza con buscar solamente el máximo.

En cada escala:

1. calculo el mapa de Template Matching;
2. tomo las posiciones que superan un porcentaje del máximo de esa escala;
3. guardo las cajas candidatas;
4. después elimino cajas repetidas con IoU.


In [ ]:
def iou(box_a, box_b):
    x1 = max(box_a[0], box_b[0])
    y1 = max(box_a[1], box_b[1])
    x2 = min(box_a[2], box_b[2])
    y2 = min(box_a[3], box_b[3])

    inter_w = max(0, x2 - x1)
    inter_h = max(0, y2 - y1)
    inter = inter_w * inter_h

    area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])

    union = area_a + area_b - inter

    if union == 0:
        return 0

    return inter / union


In [ ]:
def nms_simple(detecciones, umbral_iou=0.30):
    detecciones = sorted(
        detecciones,
        key=lambda d: d['score'],
        reverse=True
    )

    seleccionadas = []

    for det in detecciones:
        repetir = False

        for guardada in seleccionadas:
            if iou(det['box'], guardada['box']) > umbral_iou:
                repetir = True
                break

        if not repetir:
            seleccionadas.append(det)

    return seleccionadas


In [ ]:
def detectar_multiescala(gray, template_gray, proporcion_umbral=0.82):
    imagen_edges = canny_simple(gray)
    template_edges = canny_simple(template_gray)

    candidatos = []

    for scale in scales:
        temp = cv.resize(
            template_edges,
            None,
            fx=scale,
            fy=scale,
            interpolation=cv.INTER_AREA
        )

        h, w = temp.shape

        if h < 8 or w < 8:
            continue

        if h >= imagen_edges.shape[0] or w >= imagen_edges.shape[1]:
            continue

        res = cv.matchTemplate(
            imagen_edges,
            temp,
            cv.TM_CCOEFF_NORMED
        )

        max_val = float(res.max())

        # umbral relativo al mejor valor de esa escala
        threshold = proporcion_umbral * max_val

        y_coords, x_coords = np.where(res >= threshold)

        for x, y in zip(x_coords, y_coords):
            candidatos.append({
                'box': (x, y, x + w, y + h),
                'score': float(res[y, x]),
                'scale': float(scale)
            })

    return candidatos


In [ ]:
multi_path = [
    p for p in image_paths
    if os.path.basename(p) == 'coca_multi.png'
][0]

multi = cv.imread(multi_path)
multi_gray = cv.cvtColor(multi, cv.COLOR_BGR2GRAY)

candidatos = detectar_multiescala(
    multi_gray,
    template_gray,
    proporcion_umbral=0.82
)

detecciones = nms_simple(candidatos, umbral_iou=0.30)

print('Candidatos antes de NMS:', len(candidatos))
print('Detecciones después de NMS:', len(detecciones))


### Visualización

Para explorar el resultado muestro primero las detecciones con mejor score.


In [ ]:
salida = multi.copy()

for det in detecciones[:20]:
    x1, y1, x2, y2 = det['box']

    cv.rectangle(
        salida,
        (x1, y1),
        (x2, y2),
        (0, 255, 0),
        2
    )

    cv.putText(
        salida,
        f"{det['score']:.2f}",
        (x1, max(15, y1 - 5)),
        cv.FONT_HERSHEY_SIMPLEX,
        0.45,
        (0, 255, 0),
        1
    )

plt.figure(figsize=(16, 8))
plt.imshow(cv.cvtColor(salida, cv.COLOR_BGR2RGB))
plt.title('Exploración de múltiples detecciones')
plt.axis('off')
plt.show()


## 7. Conclusión de la exploración

A partir de estas pruebas dejo como base para la entrega:

- `TM_CCOEFF_NORMED`.
- búsqueda multiescala con `cv.resize`;
- preprocesamiento con Canny;
- umbral relativo al máximo de cada mapa de respuesta;
- IoU para eliminar cajas repetidas;
- misma función para todas las imágenes.

La notebook final queda más corta y evita mostrar todas estas pruebas intermedias.
